# Gate 2.5 - Preregistration Bundle And Release Blocker

Mục tiêu notebook:

- Đọc đúng Gate 2 source of truth đã pass synthetic validation.
- Tạo `prereg_bundle.json` khóa hash từ Gate 0, Gate 1, Gate 2 và `gate_threshold_registry.json`.
- Sinh `environment_lock.json`, `comparator_cards/`, `revision_policy.md`, `prereg_validation_report.md`, và `gate25_summary.json`.
- Kiểm tra bundle có `status: locked` và có `artifact_hashes` đầy đủ.
- Kiểm tra bằng guard logic rằng real-data phases chỉ được phép khi dùng đúng bundle khóa.

Giới hạn liêm chính khoa học:

- Gate 2.5 là release-control/preregistration artefact, không phải bằng chứng mô hình.
- Lock prereg bundle không chứng minh hiệu quả privileged transfer trên dữ liệu thật.
- Mọi thay đổi sau prereg có ảnh hưởng claim/comparator/threshold/teacher/control/reporting phải đi vào revision log và bị xem là post-hoc nếu không refreeze/rerun.
- Notebook chỉ orchestration qua CLI; logic bundle nằm trong repo để có test và version control.


In [ ]:
# ============================================================
# Gate 2.5 - Cell 1: Mount Drive, clone/pull repo nếu cần
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Đảm bảo repo tồn tại trong /content/eeg-ds004752.
# - Pull code mới nhất có CLI gate25.
# - Chạy unit tests trước khi tạo prereg bundle.
#
# Lưu ý:
# - Repo private cần GitHub token.
# - Token không được in ra output.
# ============================================================

from pathlib import Path
from google.colab import drive
import os
import subprocess
import getpass
import base64

# Mount Drive. Nếu đã mount, Colab sẽ báo Drive already mounted.
drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True):
    print('
$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

%cd /content/eeg-ds004752

pull_result = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull_result.returncode != 0:
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'], cwd=REPO_DIR)

# Cần có commit chứa Gate 2.5 workflow hoặc mới hơn.
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

# Chạy toàn bộ tests. Sau Gate 2.5, kỳ vọng tối thiểu là 24 tests OK.
run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Gate 2.5 - Cell 2: Khai báo Gate 2 source of truth và output
# ============================================================
# Mục tiêu:
# - Khóa đúng Gate 2 run đã synthetic-ready.
# - Khai báo output root cho prereg bundle.
# - Không tự suy diễn từ dataset/raw files.
# ============================================================

from pathlib import Path
import json

GATE2_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate2/20260418T160143330194Z')
PREREG_ROOT = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/prereg')

print('Gate 2 source of truth:', GATE2_RUN)
print('Prereg output root:', PREREG_ROOT)

if not GATE2_RUN.exists():
    raise FileNotFoundError(f'Missing Gate 2 source of truth: {GATE2_RUN}')

required_gate2_files = [
    'gate2_summary.json',
    'synthetic_generator_spec.json',
    'synthetic_recovery_report.json',
    'synthetic_recovery_report.md',
    'gate_threshold_registry.json',
]

print('
================ Gate 2 Artifact Check ================')
for name in required_gate2_files:
    path = GATE2_RUN / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing Gate 2 artifact: {path}')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 3: Preflight kiểm tra Gate 2 readiness
# ============================================================
# Mục tiêu:
# - Đảm bảo Gate 2 đã synthetic-ready.
# - Đảm bảo threshold registry đã locked_after_gate2_pass.
# - Đảm bảo Gate 2 chưa authorize real-data phase.
# ============================================================

gate2_summary = json.loads((GATE2_RUN / 'gate2_summary.json').read_text())
threshold_registry = json.loads((GATE2_RUN / 'gate_threshold_registry.json').read_text())

errors = []
if gate2_summary.get('status') != 'gate2_synthetic_ready':
    errors.append(f"Gate 2 status invalid: {gate2_summary.get('status')}")
if gate2_summary.get('recovery_status') != 'passed':
    errors.append(f"Gate 2 recovery status invalid: {gate2_summary.get('recovery_status')}")
if gate2_summary.get('threshold_registry_status') != 'locked_after_gate2_pass':
    errors.append(f"Gate 2 threshold registry status invalid: {gate2_summary.get('threshold_registry_status')}")
if gate2_summary.get('real_data_phase_authorized') is not False:
    errors.append('Gate 2 must not authorize real-data phases before Gate 2.5')
if threshold_registry.get('status') != 'locked_after_gate2_pass':
    errors.append(f"Threshold registry status invalid: {threshold_registry.get('status')}")
if threshold_registry.get('recovery_status') != 'passed':
    errors.append(f"Threshold registry recovery_status invalid: {threshold_registry.get('recovery_status')}")

if errors:
    print('Gate 2.5 preflight FAILED:')
    for item in errors:
        print('-', item)
    raise SystemExit('Stop Gate 2.5: Gate 2 source is not valid.')

print('================ Gate 2.5 Preflight ================')
print('Gate 2 status:', gate2_summary['status'])
print('Gate 0 source:', gate2_summary['gate0_source_of_truth'])
print('Gate 1 source:', gate2_summary['gate1_source_of_truth'])
print('Recovery status:', gate2_summary['recovery_status'])
print('Threshold registry status:', gate2_summary['threshold_registry_status'])
print('Threshold registry hash:', threshold_registry['threshold_registry_hash_sha256'])
print('Real-data phase authorized by Gate 2:', gate2_summary['real_data_phase_authorized'])
print('
Gate 2.5 preflight PASSED.')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 4: Kiểm tra prereg assembly config
# ============================================================
# Mục tiêu:
# - Đọc config prereg assembly trong repo.
# - Kiểm tra có registry configs, comparator configs, revision policy.
# ============================================================

config_path = REPO_DIR / 'configs/prereg/prereg_assembly.json'
if not config_path.exists():
    raise FileNotFoundError(f'Missing prereg assembly config: {config_path}')

prereg_config = json.loads(config_path.read_text())

required_top_keys = [
    'study_id',
    'version',
    'parent_doc_ids',
    'revision_policy',
    'allowed_real_phases_after_lock',
    'comparator_configs',
    'registry_configs',
]
for key in required_top_keys:
    if key not in prereg_config:
        raise KeyError(f'Missing prereg config key: {key}')

print('================ Prereg Assembly Config ================')
print('Study ID:', prereg_config['study_id'])
print('Version:', prereg_config['version'])
print('Parent docs:', prereg_config['parent_doc_ids'])
print('Allowed real phases after lock:', prereg_config['allowed_real_phases_after_lock'])
print('Comparator configs:', list(prereg_config['comparator_configs'].keys()))
print('Registry configs:', list(prereg_config['registry_configs'].keys()))
print('Revision policy:', json.dumps(prereg_config['revision_policy'], indent=2))


In [ ]:
# ============================================================
# Gate 2.5 - Cell 5: Chạy Gate 2.5 prereg assembly bằng CLI
# ============================================================
# Mục tiêu:
# - Sinh prereg_bundle.json và các artefact đi kèm.
# - Toàn bộ logic nằm trong repo, không viết bundle thủ công trong notebook.
#
# Artifact được sinh:
# - prereg_bundle.json
# - environment_lock.json
# - comparator_cards/*.json
# - revision_policy.md
# - prereg_validation_report.md
# - gate25_summary.json
# ============================================================

run([
    'python', '-m', 'src.cli', 'gate25',
    '--profile', 't4_safe',
    '--gate2-run', str(GATE2_RUN),
    '--config', 'configs/prereg/prereg_assembly.json',
    '--output-root', str(PREREG_ROOT),
], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Gate 2.5 - Cell 6: Validate gate25_summary.json
# ============================================================
# Mục tiêu:
# - Đọc latest prereg run.
# - Xác nhận release blocker satisfied.
# - Xác nhận bundle hash được ghi.
# ============================================================

LATEST_PREREG = PREREG_ROOT / 'latest.txt'
if not LATEST_PREREG.exists():
    raise FileNotFoundError(f'Missing prereg latest pointer: {LATEST_PREREG}')

prereg_run = Path(LATEST_PREREG.read_text().strip())
summary_path = prereg_run / 'gate25_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing Gate 2.5 summary: {summary_path}')

gate25_summary = json.loads(summary_path.read_text())

print('================ Gate 2.5 Summary ================')
print('Run dir:', prereg_run)
print('Status:', gate25_summary['status'])
print('Prereg bundle:', gate25_summary['prereg_bundle'])
print('Prereg bundle hash:', gate25_summary['prereg_bundle_hash_sha256'])
print('Gate 0 source:', gate25_summary['gate0_source_of_truth'])
print('Gate 1 source:', gate25_summary['gate1_source_of_truth'])
print('Gate 2 source:', gate25_summary['gate2_source_of_truth'])
print('Threshold registry hash:', gate25_summary['threshold_registry_hash_sha256'])
print('Release blocker satisfied:', gate25_summary['release_blocker_satisfied'])
print('Real-data phase authorized by prereg:', gate25_summary['real_data_phase_authorized_by_prereg'])
print('Authorized real phases:', gate25_summary['authorized_real_phases'])

assert gate25_summary['status'] == 'gate2_5_prereg_bundle_locked'
assert gate25_summary['release_blocker_satisfied'] is True
assert gate25_summary['real_data_phase_authorized_by_prereg'] is True
assert gate25_summary['prereg_bundle_hash_sha256']

print('
Gate 2.5 summary validation PASSED.')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 7: Kiểm tra prereg bundle và artifact hashes
# ============================================================
# Mục tiêu:
# - Đảm bảo prereg_bundle.json có status locked.
# - Đảm bảo có artifact_hashes cho Gate 0, Gate 1, Gate 2, threshold registry, configs.
# - Đảm bảo data freeze hash manifest/cohort lock có mặt.
# ============================================================

bundle_path = Path(gate25_summary['prereg_bundle'])
if not bundle_path.exists():
    raise FileNotFoundError(f'Missing prereg bundle: {bundle_path}')

bundle = json.loads(bundle_path.read_text())

print('================ Prereg Bundle ================')
print('Status:', bundle['status'])
print('Gate:', bundle['gate'])
print('Study ID:', bundle['study_id'])
print('Version:', bundle['version'])
print('Locked at:', bundle['locked_at'])
print('Bundle hash:', bundle['prereg_bundle_hash_sha256'])
print('Allowed real phases:', bundle['allowed_real_phases'])
print('Release blocker satisfied:', bundle['release_policy']['release_blocker_satisfied'])

assert bundle['status'] == 'locked'
assert bundle['gate'] == 'gate2_5_preregistration_bundle'
assert bundle['artifact_hashes']
assert bundle['data_freeze']['manifest_hash']
assert bundle['data_freeze']['cohort_lock_hash']
assert bundle['decision_layer']['decision_memo_hash']
assert bundle['decision_layer']['sesoi_registry_hash']
assert bundle['synthetic_validation']['threshold_registry_hash']
assert bundle['environment_lock_hash']

for group in ['gate0', 'gate1', 'gate2', 'registries_and_specs', 'comparator_configs', 'comparator_cards']:
    assert group in bundle['artifact_hashes']
    print(group, 'hash entries:', len(bundle['artifact_hashes'][group]))

print('
Prereg bundle hash validation PASSED.')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 8: Kiểm tra environment lock và comparator cards
# ============================================================
# Mục tiêu:
# - Đảm bảo environment_lock.json tồn tại.
# - Đảm bảo comparator_cards tồn tại cho comparator chính.
# ============================================================

env_path = prereg_run / 'environment_lock.json'
if not env_path.exists():
    raise FileNotFoundError(f'Missing environment lock: {env_path}')

environment_lock = json.loads(env_path.read_text())
print('================ Environment Lock ================')
print('Status:', environment_lock['status'])
print('Python executable:', environment_lock['python']['executable'])
print('Python version:', environment_lock['python']['version'].split()[0])
print('Platform:', environment_lock['platform']['platform'])
print('Git commit:', environment_lock['git']['commit'])
print('Working tree clean:', environment_lock['git']['working_tree_clean'])
print('pip freeze count:', len(environment_lock.get('pip_freeze', [])))

cards_dir = prereg_run / 'comparator_cards'
if not cards_dir.exists():
    raise FileNotFoundError(f'Missing comparator cards dir: {cards_dir}')

cards = sorted(cards_dir.glob('*.json'))
print('
================ Comparator Cards ================')
for path in cards:
    card = json.loads(path.read_text())
    print('-', path.name, '| id:', card['comparator_id'], '| status:', card['status'], '| config hash:', card['config_sha256'])

required_cards = {'A2d_riemannian.json', 'A3_distillation.json', 'A4_privileged.json', 'EEGNet.json', 'ShallowConvNet.json'}
assert required_cards.issubset({path.name for path in cards})

print('
Environment lock and comparator cards validation PASSED.')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 9: Validate guard logic without running real model
# ============================================================
# Mục tiêu:
# - Kiểm tra bundle có thể được guard đọc là locked.
# - Không chạy phase real-data model.
# - Chỉ gọi assert_real_phase_allowed để xác nhận release blocker.
# ============================================================

import sys
sys.path.insert(0, str(REPO_DIR))

from src.guards import assert_real_phase_allowed

print('================ Guard Validation ================')
for phase in ['phase05_real', 'phase1_real', 'phase2_real', 'phase3_real']:
    checked_bundle = assert_real_phase_allowed(phase, bundle_path)
    print(phase, 'guard status = allowed by locked prereg bundle, bundle hash =', checked_bundle['prereg_bundle_hash_sha256'])

print('
Guard validation PASSED. No real-data model was run in this cell.')


In [ ]:
# ============================================================
# Gate 2.5 - Cell 10: Preview validation report và revision policy
# ============================================================
# Mục tiêu:
# - Preview báo cáo validation human-readable.
# - Preview revision policy.
# ============================================================

validation_report_path = prereg_run / 'prereg_validation_report.md'
revision_policy_path = prereg_run / 'revision_policy.md'

print('================ Prereg Validation Report Preview ================')
print(validation_report_path.read_text()[:4000])

print('
================ Revision Policy Preview ================')
print(revision_policy_path.read_text()[:2500])


In [ ]:
# ============================================================
# Gate 2.5 - Cell 11: Kết luận đúng phạm vi
# ============================================================
# Mục tiêu:
# - In kết luận không overclaim.
# - Ghi source of truth cho Gate 2.5.
# ============================================================

print('================ Interpretation ================')
print('OK: Gate 2.5 preregistration bundle is locked.')
print('OK: Gate 0, Gate 1, Gate 2, threshold registry, config registries, comparator cards, and environment lock are hash-linked.')
print('OK: Release blocker is satisfied for guarded real-data phases using this exact bundle.')
print('NOT OK TO CLAIM: This does not prove real EEG model efficacy or privileged transfer.')
print('REQUIRED: Any post-prereg claim-affecting change must go to revision log and be treated as post-hoc unless refrozen/rerun.')
print('
Gate 2.5 source of truth:', prereg_run)
print('Prereg bundle:', bundle_path)
print('Prereg bundle hash:', bundle['prereg_bundle_hash_sha256'])
